# 🏗️ Notebook: Introduction to Structured Outputs & Validation with LLMs

In this notebook you will learn how to use Large Language Models (LLMs) to generate structured outputs.

## 📚 Sources

- [OpenAI: Validating LLM Outputs](https://platform.openai.com/docs/guides/structured-outputs?api-mode=chat)
- [Pydantic Documentation](https://pydantic.dev/)
- [Ollama Structured Outputs](https://ollama.com/blog/structured-outputs)

---

Good luck with experimenting and validating! 🤗

In [14]:
%%capture
!pip install pydantic

In [15]:
# Import the required libraries
from pydantic import BaseModel, Field
from openai import OpenAI
from typing import List, Literal, Optional
import json

In [ ]:
LLM_URL = "http://localhost:11434"
LLM_MODEL = "gemma3:4b"

In [17]:
client = OpenAI(
    base_url=LLM_URL,
    api_key="ollama",
)

A Gemma3 model is running in the backend, accessed via Ollama's OpenAI-compatible API. **Reminder**: Besides Ollama, there are other providers that offer OpenAI-compatible APIs, e.g., [vLLM](https://docs.vllm.ai/en/v0.8.2/features/structured_outputs.html).

**Structured outputs** enable restricting the output of a model to a specific format defined by a **JSON schema**. Ollama currently supports structured outputs for JSON format, as do OpenAI's endpoints for their GPT models. The VLLM library, which also allows us to use open-source LLMs, supports regex as well, although there are still some bugs (as of October 2025).

 
Use cases for structured outputs:

- Extract data from documents
- Extract data from images
- Structure all language model responses

## Interlude: JSON – JavaScript Object Notation

**JSON** is a simple, text-based format for storing and exchanging data.  
It is supported by many programming languages and is especially popular for web APIs.

### Basic Principles
- Data is stored as **key-value pairs**.
- Structures can be **nested** (objects within objects, lists within objects).

### Data Types in JSON
- **String**: Text in quotes, e.g., `"Hello"`
- **Number**: Integers or decimals, e.g., `42` or `3.14`
- **Boolean**: Truth values `true` or `false`
- **Null**: Empty value `null`
- **Array**: List of values, e.g., `[1, 2, 3]`. Different data types are also possible, e.g., `["Text", 42, true]`
- **Object**: Collection of key-value pairs, e.g., `{"name": "Max", "age": 30}`

### Example
```json
{
  "name": "Bello",
  "tier": "Hund",
  "alter": 5,
  "spielzeug": ["Ball", "Seil"],
  "geimpft": true,
  "besitzer": null
}

In [18]:
# Define a model for a pet
# name: str       -> Attribute 'name' of type String
# species: str     -> Attribute 'species' of type String (type of animal)
# age: int        -> Attribute 'age' of type Integer
# color: str | None -> Optional attribute 'color', can be String or None
# favorite_toy: str | None -> Optional attribute 'favorite_toy', String or None
# BaseModel from Pydantic automatically generates the constructor, validation and attribute access
class Pet(BaseModel):
    name: str
    species: str
    age: int
    color: str | None
    favorite_toy: str | None

# Define a model for a list of pets
# pets: list[Pet] -> Attribute 'pets' is a list of Pet objects
class PetList(BaseModel):
    pets: list[Pet]


prompt = '''
I have two pets.
A cat named Luna who is 5 years old and loves playing with yarn. She has grey fur.
I also have a 2 year old black cat named Loki who loves tennis balls.
'''

completion = client.chat.completions.parse(
    temperature=0,
    model=LLM_MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ],
    response_format=PetList,
)

output = completion.choices[0].message.content

# The LLM response is in string format. We need to convert it to a Python dictionary.
output_dict = json.loads(output)
print(output_dict)

{'pets': [{'name': 'Luna', 'species': 'cat', 'age': 5, 'color': 'grey', 'favorite_toy': 'yarn'}, {'name': 'Loki', 'species': 'cat', 'age': 2, 'color': 'black', 'favorite_toy': 'tennis balls'}]}


### Example 2: Text Classification

We can also define a simple classification schema to categorize texts. In this example, we classify short texts into the categories "positive", "negative" or "neutral".

In [19]:
class SentimentResult(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"] # Literal defines allowed values for the attribute 'sentiment'

prompt = f"Classify the sentiment of this text as positive, neutral, or negative:\n\nI love programming in Python!"

completion = client.chat.completions.parse(
    temperature=0,
    model=LLM_MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ],
    response_format=SentimentResult,
)

output = completion.choices[0].message.content

# The LLM response is in string format, since LLMs output text. We need to convert the response to a Python dictionary.
output_dict = json.loads(output)
print(f"Output Dict: {output_dict}")

# Let's output only the sentiment
print(f"Sentiment: {output_dict['sentiment']}")

Output Dict: {'sentiment': 'positive'}
Sentiment: positive


### Example 3: JSON Schema in the Prompt

We can also pass the json schema directly to the LLM in the prompt. `json.dumps(json_schema, indent=2)` converts the Python dictionary into a JSON string that can be used in the prompt. `indent=2` ensures readable formatting with indentation. 2 means that each level in the JSON is indented by 2 spaces.

In [20]:
# Define a Pydantic class for a single software developer
class Developer(BaseModel):
    name: str  # Name of the developer
    programming_language: Literal["python", "java", "javascript", "csharp", "cpp", "go", "rust", "php"] 
    experience_years: int = Field(ge=0, le=50)  # The developer has between 0 and 50 years of experience. Field enables validations
    specialization: Literal["frontend", "backend", "fullstack", "data_science", "devops", "mobile"]

# Define a Pydantic class for a list of exactly 2 developers
class DeveloperList(BaseModel):
    developers: List[Developer] = Field(min_length=2, max_length=2)  # Exactly 2 Developer objects

# Generate the JSON schema from the DeveloperList class
json_schema = DeveloperList.model_json_schema()

prompt = f"""
Generate a JSON array of exactly 2 software developers with the following fields:
- name: string
- programming_language: one of ["python", "java", "javascript", "csharp", "cpp", "go", "rust", "php"]
- experience_years: integer between 0 and 50
- specialization: one of ["frontend", "backend", "fullstack", "data_science", "devops", "mobile"]      
Make sure the output strictly adheres to this JSON schema:
{json.dumps(json_schema, indent=2)}
"""

completion = client.chat.completions.parse(
    temperature=0,
    model=LLM_MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ],
    response_format=DeveloperList,
)

output = completion.choices[0].message.content

# The LLM response is in string format. We need to convert it to a Python dictionary.
output_dict = json.loads(output)
print(output_dict)

{'developers': [{'name': 'Alice Johnson', 'programming_language': 'python', 'experience_years': 10, 'specialization': 'data_science'}, {'name': 'Bob Williams', 'programming_language': 'javascript', 'experience_years': 35, 'specialization': 'frontend'}]}


### Exercise 1

To practice validating JSON outputs, create a prompt that instructs the LLM to deliver a structured response in JSON format.

---

#### 🎯 Task

Define a simple JSON schema to generate dummy data for **musicians**.

#### 🎤 Artist Schema

| Field    | Description                                          |
| -------- | ---------------------------------------------------- |
| `name`   | Name of the artist                                   |
| `age`    | Age (between 18 and 100)                             |
| `genre`  | Music genre from: **Rock**, **Pop**, **Jazz**, **Classical** |
| `albums` | List of albums                                       |

#### 💿 Album Schema

| Field           | Description                         |
| --------------- | ----------------------------------- |
| `title`        | Album title                         |
| `release_year` | Release year (between 1900 and 2023) |
| `tracks`       | List of song titles                 |

---

#### 💡 Tip

Use Pydantic classes with `Field()` validations and the `Literal` type system for genre selection!

<details>
<summary><b>Show Solution</b></summary>

```python
# Define a Pydantic class for an album
class Album(BaseModel):
    title: str
    release_year: int = Field(ge=1900, le=2023)
    tracks: List[str]

# Define a Pydantic class for a musician
class Musician(BaseModel):
    name: str
    age: int = Field(ge=18, le=100)
    genre: Literal["Rock", "Pop", "Jazz", "Classical"]
    albums: List[Album]

# Define a Pydantic class for a list of musicians
class MusicianList(BaseModel):
    musicians: List[Musician] = Field(min_length=2, max_length=2)

# Generate the JSON schema from the MusicianList class
json_schema = MusicianList.model_json_schema()

prompt = f"""
Generate a list of 2 musicians with the following fields:
- name: string
- age: integer between 18 and 100
- genre: one of ["Rock", "Pop", "Jazz", "Classical"]
- albums: list of albums, each album has:
  - title: string
  - release_year: integer between 1900 and 2023
  - tracks: list of song titles (strings)

Return the data as JSON that strictly conforms to this schema:
{json.dumps(json_schema, indent=2)}
"""

completion = client.beta.chat.completions.parse(
    temperature=0,
    model=LLM_MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ],
    response_format=MusicianList,
)

output = completion.choices[0].message.content

# The LLM response is in string format. We need to convert it to a Python dictionary.
output_dict = json.loads(output)
print(json.dumps(output_dict, indent=2, ensure_ascii=False))
```

</details>

In [21]:
### Your code...

### Example 4: Guided JSON for Structured Extraction of Information from Documents
Example for structured extraction of information from an image.

Example:

<img src="../content/billing.png" alt="Purchase Order Example" width="500">

In [22]:
# Let's load the image and encode it in Base64. Base64 is a text format that converts binary data (like images) into a text representation.
import base64
with open("../content/billing.png", "rb") as image_file:
    # Convert file to Base64
    encoded_string = base64.b64encode(image_file.read()).decode("utf-8")

# Format as data URL. Data URLs allow embedding images directly in HTML or JSON.
data_url = f"data:image/png;base64,{encoded_string}"

In [23]:
# Let's start without JSON validation to test the output.
# The LLM should first be able to analyze an image and generate a response.
prompt = "What do you see in this image?"
completion = client.chat.completions.parse(
    temperature=0,
    model=LLM_MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {
                    "type": "image_url",
                    "image_url": data_url
                },
            ],
        }
    ]
)

print(completion.choices[0].message.content)

Here's a breakdown of what I see in the image – it’s a purchase order:

**Header:**

*   **Company Name:** BIZZLIBRARY.COM

**Vendor Information:**

*   **Vendor Name:** ABC Office Supplies
*   **Address:** 1687 K Street NW, Washington DC 2006
*   **Contact No.:** (203) 743-8983

**Customer Information:**

*   **Customer Name:** Franklin Middle School
*   **Address:** Washington DC, USA
*   **Contact No.:** (203) 334-7234

**Sales Information:**

*   **Sales Person:** Helen Wilson on Purchasing Department

**Items Ordered:**

*   **Item 1:** Pencils HB - Dozen 5 - $50.00
*   **Item 2:** Pencils 2B - Dozen 4 - $40.00
*   **Item 3:** Paper - A4, Photo copier, 70 gram - Ream 10 - $3.00
*   **Item 4:** Paper - A4, Photo copier, 80 gram - Ream 15 - $3.20
*   **Item 5:** Pen - Ball Point, Blue - Box 10 - $10.00
*   **Item 6:** Highlighter - 3 color - Sets 8 - $8.00

**Additional Notes:**

*   Payment shall be 30 days upon delivery of the above items.

**Totals:**

*   Subtotal: $316.00
*   T

### 4. Guided JSON for Structured Extraction of Information from Documents

Now we will look at a more complex example: structured extraction of information from a receipt.

#### Goal of the Exercise

We want to analyze a restaurant receipt and automatically extract the most important information:
- **List of items** ordered with details
- **Total amount** of the bill

#### 🖼️ The Document

This is what the receipt to be analyzed looks like:

<img src="../content/receipt.png" alt="Receipt Example" width="400">

In [24]:
with open("../content/receipt.png", "rb") as image_file:
    encoded_string = base64.b64encode(image_file.read()).decode("utf-8")
data_url = f"data:image/png;base64,{encoded_string}"

In [29]:
class ReceiptItem(BaseModel):
    description: str = Field(min_length=1, max_length=10000)
    quantity: int = Field()
    price_usd: float = Field()

class Receipt(BaseModel):
    total_amount: float = Field(gt=0)
    items: List[ReceiptItem] = Field(min_length=1)

In [30]:
prompt = "Analyze the receipt in the image and give me the details as JSON. Extract all items with their descriptions, quantities, and prices: "
completion = client.chat.completions.parse(
    temperature=0,
    model=LLM_MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                },
                {
                    "type": "image_url",
                    "image_url": data_url
                },
            ],
        }
    ],
    response_format=Receipt,
)

print(completion.choices[0].message.content)

{
  "total_amount": 157.51,
  "items": [
    {
      "description": "Shrimp Scampi",
      "quantity": 1,
      "price_usd": 27.00
    },
    {
      "description": "Chicken Milanese",
      "quantity": 2,
      "price_usd": 46.00
    },
    {
      "description": "Veal Milanese",
      "quantity": 1,
      "price_usd": 27.00
    },
    {
      "description": "Grey Goose",
      "quantity": 1,
      "price_usd": 30.00
    },
    {
      "description": "Pineapple Juice",
      "quantity": 1,
      "price_usd": 3.00
    },
    {
      "description": "Belvedere",
      "quantity": 1,
      "price_usd": 12.00
    },
    {
      "description": "Becks Non Alcoholic",
      "quantity": 1,
      "price_usd": 6.00
    },
    {
      "description": "Corona",
      "quantity": 1,
      "price_usd": 6.00
    },
    {
      "description": "Comp Item",
      "quantity": 1,
      "price_usd": -12.00
    }
  ]
}



### Exercise 2

Now it's your turn! You want to automatically extract information from the purchase order. 

Example:

<img src="../content/billing.png" alt="Purchase Order Example" width="500">

The LLM should return the following structured information in JSON format:

* Vendor information:
  - Vendor name
  - Address
  - Contact number
  - Email address

* Customer information:
  - Customer name
  - Address
  - Contact number
  - Email address

  - Total amount

* Purchase order details:  - Discount percentage and amount

  - Purchase order number  - Tax percentage and amount

  - Date  - Subtotal

* Financial summary:

* List of items, each with:

  - Item description  - Total price

  - Unit  - Unit price
  - Quantity

<details>
<summary><b>Show Solution</b></summary>

```python
# Define a Pydantic class for contact information
class ContactInfo(BaseModel):
    name: str
    address: str
    contact_number: Optional[str] = None
    email_address: Optional[str] = None

# Define a Pydantic class for an order item
class OrderItem(BaseModel):
    description: str
    unit: str
    quantity: int = Field(ge=0)
    unit_price: float = Field(ge=0)
    total_price: float = Field(ge=0)

# Define a Pydantic class for financial summary
class FinancialSummary(BaseModel):
    subtotal: float = Field(ge=0)
    tax_percentage: float = Field(ge=0, le=100)
    tax_amount: float = Field(ge=0)
    discount_percentage: float = Field(ge=0, le=100)
    discount_amount: float = Field(ge=0)
    total: float = Field(ge=0)

# Define a Pydantic class for the purchase order
class PurchaseOrder(BaseModel):
    vendor: ContactInfo
    customer: ContactInfo
    purchase_order_number: str
    date: str
    items: List[OrderItem] = Field(min_length=1)
    financial_summary: FinancialSummary

# Let's load the image and encode it in Base64
with open("../content/billing.png", "rb") as image_file:
    encoded_string = base64.b64encode(image_file.read()).decode("utf-8")
data_url = f"data:image/png;base64,{encoded_string}"

prompt = "Analyze the purchase order in the image and extract all information including vendor details, customer details, order items, and financial summary as JSON."

completion = client.chat.completions.parse(
    temperature=0,
    model=LLM_MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {
                    "type": "image_url",
                    "image_url": data_url
                },

            ],
        }
    ],
    response_format=PurchaseOrder
)
output_dict = json.loads(output)
print(json.dumps(output_dict, indent=2, ensure_ascii=False))

# The LLM response is in string format. We need to convert it to a Python dictionary.

output = completion.choices[0].message.content
```

In [ ]:
# Your code goes here...